# Genetic Algorithm Assignment
30% of the overall grade for Artificial Intelligence model


Emmanuella G. Nyamekye

## The Problem

The goal of this project is to solve the Portfolio Optimization Problem using 10 different assets from the FTSE 100.

Find the best mix or "Sweet Spot" of these 10 stocks so you get the highest reward for the lowest amount of risk.

The specific implementation uses historical data from the past year to calculate the Sharpe Ratio for thousands of potential asset combinations. The "Individual" in this algorithm is a set of 10 percentages (weights) that must sum to 100%.

---

## Discussion of the suitability of Genetic Algorithms

Genetic Algorithms are suited for Portfolio Optimization for the following reasons:


*   Non-Linear Search Space: The relationship between 10 different assets is not a straight line.
*   Constraint Handling: Normalization ensures that the sum of weights always equals 1.0. GAs allow to "evolve" solutions that stay within these constraints.
*   High Dimensionality: Balancing 10 assets involves a 10-dimensional space. GAs use Crossover (combining successful traits of two portfolios) and Mutation (randomly testing new allocations) to search this high-dimensional space much faster.

---

In [1]:
import yfinance as yf
import numpy as np
from copy import deepcopy

In [2]:
# 10 Diversified FTSE 100
ftse_assets = [
    'SHEL.L',
    'AZN.L',
    'HSBA.L',
    'ULVR.L',
    'BP.L',
    'GSK.L',
    'REL.L',
    'DGE.L',
    'RIO.L',
    'LSEG.L'
]

In [3]:
# 1 year of historical data
data = yf.download(ftse_assets, period="1y")['Close']
returns_data = data.pct_change().dropna()

/tmp/ipykernel_11330/538728138.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ftse_assets, period="1y")['Close']
[*********************100%***********************]  10 of 10 completed
/tmp/ipykernel_11330/538728138.py:3: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_data = data.pct_change().dropna()


# The problem and the cost function

In [4]:
class PortfolioProblem:
    def __init__(self, returns):
        self.returns = returns
        self.number_of_genes = len(ftse_assets)
        self.min = 0.0 # Weights cannot be negative
        self.max = 1.0

    def normalize(self, weights):
        # Make sure the portfolio sum = 1.0 (100%).
        return weights / np.sum(weights) # Normalizing a list of numbers

In [5]:
# Sharpe Ratio to find the best balance of return vs. risk
def calculate_portfolio_cost(weights, returns):
    # 1. Annualized Portfolio Return
    port_return = np.sum(returns.mean() * weights) * 252

    # 2. Annualized Portfolio Risk (Standard Deviation)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 252, weights)))

    if port_vol == 0: return 0

    # 3. Calculate Sharpe Ratio
    sharpe_ratio = port_return / port_vol

    # Return it as a negative value.
    return -sharpe_ratio

# The Individual

*   Chromosone
*   Crossover
*   Mutation

In [6]:
class Individual:
    def __init__(self, prob):
        # Create random weights and normalize to 100%
        raw_weights = np.random.uniform(prob.min, prob.max, prob.number_of_genes)
        self.chromosome = prob.normalize(raw_weights)
        self.cost = calculate_portfolio_cost(self.chromosome, prob.returns)

    def Crossover(self, other, epsilon):
        # Standard Linear Crossover
        child1 = deepcopy(self)
        child2 = deepcopy(other)
        alpha = np.random.uniform(-epsilon, 1 + epsilon)

        child1.chromosome = alpha * self.chromosome + (1 - alpha) * other.chromosome
        child2.chromosome = (1 - alpha) * self.chromosome + alpha * other.chromosome

        # Ensure children are valid (0-1 range and sum to 1.0)
        for child in [child1, child2]:
            child.chromosome = np.clip(child.chromosome, 0.001, 1.0) # Clip makes sure chromosome is between 0.001 and 1.0
            child.chromosome = child.chromosome / np.sum(child.chromosome)

        return child1, child2

    def Mutate(self, mutation_rate, mutation_range):
        for i in range(len(self.chromosome)):
            if np.random.uniform(0, 1) < mutation_rate:
                # Randomly move the weight up or down
                self.chromosome[i] += np.random.normal(0, mutation_range)

        # Re-normalize to fix the "Total Weight Error"
        self.chromosome = np.clip(self.chromosome, 0.001, 1.0)
        self.chromosome = self.chromosome / np.sum(self.chromosome)

## Discussion and justification on the approaches taken for the above


Each Individual represents one possible portfolio, a set of 10 weights, one per FTSE 100 asset, that must always sum to 1.0 (100%).

```
raw_weights = np.random.uniform(prob.min, prob.max, prob.number_of_genes)
```

Each Individual starts with completely random weights between 0 and 1. This ensures the initial population is different. If all individuals started with the same weight, the algorithm would struggle to explore different portfolio mixes.

```
self.chromosome = prob.normalize(raw_weights)
```

Raw random weights won't sum to 1.0, so we divide each weight by the total sum.

```
alpha = np.random.uniform(-epsilon, 1 + epsilon)
child1.chromosome = alpha * self.chromosome + (1 - alpha) * other.chromosome
child2.chromosome = (1 - alpha) * self.chromosome + alpha * other.chromosome
```

Linear crossover blends the weights of two parents using a random mixing factor alpha. This produces children whose weights are a weighted average of both parents.
The epsilon parameter allows alpha to slightly exceed the range [0, 1], meaning children can occasionally explore slightly outside the range between their parents. This makes sure that the population does not lose diversity.


```
self.chromosome[i] += np.random.normal(0, mutation_range)
```

Mutation randomly moves individual weights, preventing the algorithm from getting stuck in a 'good enough' spot.

It works by making a small random adjustment to a weight, moving it either slightly up or slightly down. Most of the time the move is tiny, but occasionally a bigger change happens. This balance is important: too many big changes would destroy good portfolios, but too few changes would cause the algorithm to stop exploring and settle for a solution that might not be the best.

# Running the algorithm


In [7]:
# Parameter class
class Parameters:
    def __init__(self):
        self.population = 60 # 60 different random portfolios
        self.maximum_number_of_generations = 100 # Rounds of evolution
        self.mutation_rate = 0.2 # Enough mutation to keep exploring, if too low the algorithm barely changes, and if too high every weight changes every round
        self.mutation_range = 0.05 # +- 5 %
        self.crossover_exploration = 0.1 # Allows childern to explore 0.1 outside of parent

In [8]:
# Run Genetic method
def run_portfolio_ga():
    prob = PortfolioProblem(returns_data)
    params = Parameters()

    # Initialize Population
    population = []
    for i in range(params.population):
        population.append(Individual(prob))
    history = []

    for gen in range(params.maximum_number_of_generations):
        children = []
        while len(children) < params.population:
            # Selection of two random parents from the population
            p1, p2 = np.random.choice(population, 2)
            # Parents combine their weights to produce two children
            c1, c2 = p1.Crossover(p2, params.crossover_exploration)
            c1.Mutate(params.mutation_rate, params.mutation_range)
            c2.Mutate(params.mutation_rate, params.mutation_range)

            # Re-calculate costs -> how good is this new portfolio
            c1.cost = calculate_portfolio_cost(c1.chromosome, prob.returns)
            c2.cost = calculate_portfolio_cost(c2.chromosome, prob.returns)
            children.extend([c1, c2])

        # Merge and keep best 60
                    # Sort (combined population + children)[keep first 60 which are the best ones]
        population = sorted(population + children, key=lambda x: x.cost)[:params.population]
        # Sharpe Ratio is negative so has to be flipped back to positive before being added to history
        history.append(-population[0].cost)

    return population[0], history

In [9]:
# Running of the algorithm with outputs
# Best individual & Best portfolio
best_ind, conv_history = run_portfolio_ga()

print(f"Optimal Sharpe Ratio: {-best_ind.cost:.2f}")
print("-" * 30)
for asset, weight in zip(ftse_assets, best_ind.chromosome):
    print(f"{asset:8}: {weight*100:5.2f}%")

Optimal Sharpe Ratio: 3.23
------------------------------
SHEL.L  :  5.33%
AZN.L   : 30.73%
HSBA.L  :  0.10%
ULVR.L  :  9.33%
BP.L    : 25.19%
GSK.L   :  0.10%
REL.L   :  0.10%
DGE.L   : 28.93%
RIO.L   :  0.10%
LSEG.L  :  0.10%


# Results and Conclusions
The Genetic Algorithm converged on a Sharpe Ratio of 3.23.
The algorithm concentrated the portfolio into three assets: AZN.L (30.73%), DGE.L (28.93%), and BP.L (25.19%), which together account for around 85% of the portfolio. The remaining seven assets were allocated the minimum possible weight (0.10%), meaning the GA dismissed them as poor contributors.

The result does vary slightly between runs due to the randomness in the GA (random initialisation, crossover, and mutation). A previous run produced a Sharpe Ratio of 3.40 with a similar but not identical allocation, which confirms the algorithm is consistently finding high-quality solutions in the same assets.

**Limitations:** The model is trained on only one year of historical data, so past performance does not guarantee future results. A better approach would be to test across multiple time periods. Additionally, the minimum weight of 0.10% prevents full asset elimination.